# Mercari(일본 중고거래 플랫폼) 가격 예측 프로젝트

## 프로젝트 흐름  
1. 환경 설정 및 데이터 로드
2. 데이터 전처리 (카테고리 분리, 결측치 처리)
3. 텍스트 특징 추출 (NLP_자연어 처리)
4. 새로운 피처 생성
5. 모델 학습
6. 가격 예측 & 제출

### 1. 환경설정 및 데이터 로드

In [1]:
pip install pandas numpy scikit-learn lightgbm nltk scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. 환경설정
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# 1. 데이터 로드
train = pd.read_csv('train.tsv', sep='\t')
test  = pd.read_csv('test.tsv',  sep='\t')

print(train.shape)
print(train.head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


(1482535, 8)
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name  price  \
0                                  Men/Tops/T-shirts        NaN   10.0   
1  Electronics/Computers & Tablets/Components & P...      Razer   52.0   
2                        Women/Tops & Blouses/Blouse     Target   10.0   
3                 Home/Home Décor/Home Décor Accents        NaN   35.0   
4                            Women/Jewelry/Necklaces        NaN   44.0   

   shipping                                   item_description  
0         1                                 No des

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1482535 entries, 0 to 1482534
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1482535 non-null  int64  
 1   name               1482535 non-null  object 
 2   item_condition_id  1482535 non-null  int64  
 3   category_name      1476208 non-null  object 
 4   brand_name         849853 non-null   object 
 5   price              1482535 non-null  float64
 6   shipping           1482535 non-null  int64  
 7   item_description   1482529 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 90.5+ MB


In [4]:
train.describe()

,train_id,item_condition_id,price,shipping
count,1.482535e+06,1.482535e+06,1.482535e+06,1.482535e+06
mean,7.412670e+05,1.907380e+00,2.673752e+01,4.472744e-01
std,4.279711e+05,9.031586e-01,3.858607e+01,4.972124e-01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,3.706335e+05,1.000000e+00,1.000000e+01,0.000000e+00
50%,7.412670e+05,2.000000e+00,1.700000e+01,0.000000e+00
75%,1.111900e+06,3.000000e+00,2.900000e+01,1.000000e+00
max,1.482534e+06,5.000000e+00,2.009000e+03,1.000000e+00


In [5]:
# 결측치 확인
train.isna().sum()

train_id                  0
name                      0
item_condition_id         0
category_name          6327
brand_name           632682
price                     0
shipping                  0
item_description          6
dtype: int64

In [7]:
# 결측치 비율 확인
train.isna().mean()

train_id             0.000000
name                 0.000000
item_condition_id    0.000000
category_name        0.004268
brand_name           0.426757
price                0.000000
shipping             0.000000
item_description     0.000004
dtype: float64

In [8]:
train['item_condition_id'].unique() # 수치형 컬럼이지만 상태의 등급을 수치로 표현

array([3, 1, 2, 4, 5])

In [9]:
train['shipping'].unique() # 수치형 컬럼이지만 배송 유무에 대한 것을 수치로 표현

array([1, 0])

### 2. category_name 분리 ('/' 구분) 및 데이터 전처리 (결측치, 이상치)

In [9]:
train['category_name'].nunique()

1287

In [10]:
train['category_name'].unique()

array(['Men/Tops/T-shirts',
       'Electronics/Computers & Tablets/Components & Parts',
       'Women/Tops & Blouses/Blouse', ..., 'Handmade/Jewelry/Clothing',
       'Vintage & Collectibles/Supplies/Ephemera',
       'Handmade/Pets/Blanket'], shape=(1288,), dtype=object)

#### 2-1.category_name 분리 : 3단계 분리

In [23]:
def split_category(category):
    """카테고리를 최대 3단계로 분리"""
    if pd.isnull(category):
        return pd.Series(['Unknown', 'Unknown', 'Unknown'])
    parts = category.split('/', 2)  # 최대 3개로 분리
    while len(parts) < 3:
        parts.append('Unknown')
    return pd.Series(parts[:3])

# 카테고리 분리 적용
train[['cat_1', 'cat_2', 'cat_3']] = train['category_name'].apply(split_category)
test[['cat_1',  'cat_2', 'cat_3']] = test['category_name'].apply(split_category)

print(train['cat_1'].value_counts().head(10))  # 상위 카테고리 확인

cat_1
Women                     664385
Beauty                    207828
Kids                      171689
Electronics               122690
Men                        93680
Home                       67871
Vintage & Collectibles     46530
Other                      45351
Handmade                   30842
Sports & Outdoors          25342
Name: count, dtype: int64


In [24]:
train.head(5)

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,cat_1,cat_2,cat_3
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet,Men,Tops,T-shirts
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...,Electronics,Computers & Tablets,Components & Parts
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...,Women,Tops & Blouses,Blouse
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...,Home,Home Décor,Home Décor Accents
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity,Women,Jewelry,Necklaces


In [25]:
# 중분류 카테고리 확인
train['cat_2'].unique()

array(['Tops', 'Computers & Tablets', 'Tops & Blouses', 'Home Décor',
       'Jewelry', 'Other', 'Swimwear', 'Apparel', 'Collectibles',
       'Makeup', 'Fragrance', 'Dresses', 'Office supplies', 'Shoes',
       'Gear', 'Athletic Apparel', 'Cell Phones & Accessories', 'Jeans',
       'Underwear', 'Skin Care', 'Toys', "Women's Handbags",
       'Video Games & Consoles', 'Coats & Jackets', 'Pants', 'Girls (4+)',
       'Antique', 'Kitchen & Dining', 'Sweaters', 'Boys 0-24 Mos',
       'Girls 0-24 Mos', 'Maternity', 'Bedding', 'Exercise',
       'Trading Cards', 'Boys (4+)', 'Storage & Organization', 'Fan Shop',
       'Girls 2T-5T', "Men's Accessories", 'Boys 2T-5T',
       "Women's Accessories", 'Daily & Travel items', 'Unknown', 'Skirts',
       'Hair Care', 'Pet Supplies', 'Book', 'Tools & Accessories',
       'Team Sports', 'Home Appliances', 'Accessories', 'Bags and Purses',
       'Sweats & Hoodies', 'Shorts', 'TV, Audio & Surveillance',
       'Outdoors', 'Bath & Body', 'Car Seats

In [26]:
train['cat_2'].nunique()

114

In [27]:
train['cat_3'].unique()

array(['T-shirts', 'Components & Parts', 'Blouse', 'Home Décor Accents',
       'Necklaces', 'Other', 'Two-Piece', 'Girls', 'Doll', 'Face',
       'Women', 'Above Knee, Mini', 'School Supplies', 'Boots',
       'Makeup Sets', 'Eyes', 'Backpacks & Carriers', 'Makeup Palettes',
       'Tank, Cami', 'Sports Bras', 'Cell Phones & Smartphones',
       'Chargers & Cradles', 'T-Shirts', 'Athletic',
       'Cases, Covers & Skins', 'Pants, Tights, Leggings', 'One-Piece',
       'Boot Cut', 'Bras', 'Stuffed Animals & Plush', 'Totes & Shoppers',
       'Shirts & Tops', 'Consoles', 'Glass', 'Vest', 'Arts & Crafts',
       'Capris, Cropped', 'Messenger & Crossbody', 'Shoes',
       'Collectibles', 'Coffee & Tea Accessories', 'Brooch', 'Headsets',
       'Rings', 'Shorts', 'Fleece Jacket', 'Dolls & Accessories',
       'Crewneck', 'Jackets', 'Home Fragrance', 'Accessories',
       'Tops & Blouses', 'Sheets & Pillowcases', 'Fitness technology',
       'Dress Up & Pretend Play', 'Animation',
       'J

In [28]:
train['cat_3'].nunique()

872

#### 2-2. category_name 결측치 확인 및 처리  

##### 2-2-1. category_name 결측치 확인

In [29]:
# 전체 결측치 현황 한눈에 보기
print("=== 결측치 현황 ===")
print(train.isna().sum())
print(f"\ncategory_name 결측치 수: {train['category_name'].isna().sum()}")
print(f"전체 대비 비율: {train['category_name'].isna().mean()*100:.2f}%")

# 결측치 샘플 확인 - 어떤 상품들인지 보기
null_category = train[train['category_name'].isna()][
    ['name', 'brand_name', 'item_description', 'price']
].sample(20, random_state=42)

print("\n=== category_name 결측 샘플 ===")
print(null_category.to_string())

=== 결측치 현황 ===
train_id                  0
name                      0
item_condition_id         0
category_name          6327
brand_name           632682
price                     0
shipping                  0
item_description          6
cat_1                     0
cat_2                     0
cat_3                     0
dtype: int64

category_name 결측치 수: 6327
전체 대비 비율: 0.43%

=== category_name 결측 샘플 ===
                                             name      brand_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

##### 2-2-2. category_name 결측치 처리 : 대분류 카테고리 키워드 매핑

In [30]:
print(sorted(train[train['cat_1'] == 'Women']['cat_2'].unique()))

['Athletic Apparel', 'Coats & Jackets', 'Dresses', 'Jeans', 'Jewelry', 'Maternity', 'Other', 'Pants', 'Shoes', 'Skirts', 'Suits & Blazers', 'Sweaters', 'Swimwear', 'Tops & Blouses', 'Underwear', "Women's Accessories", "Women's Handbags"]


In [32]:
print(sorted(train[train['cat_1'] == 'Beauty']['cat_2'].unique()))

['Bath & Body', 'Fragrance', 'Hair Care', 'Makeup', 'Other', 'Skin Care', 'Tools & Accessories']


In [33]:
print(sorted(train[train['cat_1'] == 'Kids']['cat_2'].unique()))

['Bathing & Skin Care', 'Boys (4+)', 'Boys 0-24 Mos', 'Boys 2T-5T', 'Car Seats & Accessories', 'Diapering', 'Feeding', 'Gear', 'Girls (4+)', 'Girls 0-24 Mos', 'Girls 2T-5T', 'Health & Baby Care', 'Nursery', 'Other', 'Potty Training', 'Pregnancy & Maternity', 'Safety', 'Strollers', 'Toys']


In [34]:
print(sorted(train[train['cat_1'] == 'Electronics']['cat_2'].unique()))

['Cameras & Photography', 'Car Audio, Video & GPS', 'Cell Phones & Accessories', 'Computers & Tablets', 'Media', 'Other', 'TV, Audio & Surveillance', 'Video Games & Consoles']


In [35]:
print(sorted(train[train['cat_1'] == 'Men']['cat_2'].unique()))

['Athletic Apparel', 'Blazers & Sport Coats', 'Coats & Jackets', 'Jeans', "Men's Accessories", 'Other', 'Pants', 'Shoes', 'Shorts', 'Suits', 'Sweaters', 'Sweats & Hoodies', 'Swimwear', 'Tops']


In [36]:
print(sorted(train[train['cat_1'] == 'Home']['cat_2'].unique()))

['Artwork', 'Bath', 'Bedding', 'Cleaning Supplies', 'Furniture', 'Home Appliances', 'Home Décor', "Kids' Home Store", 'Kitchen & Dining', 'Other', 'Seasonal Décor', 'Storage & Organization']


In [ ]:
print(sorted(train[train['cat_1'] == 'Vintage & Collectibles']['cat_2'].unique()))

['Accessories', 'Antique', 'Bags and Purses', 'Book', 'Clothing', 'Collectibles', 'Electronics', 'Furniture', 'Home Decor', 'Housewares', 'Jewelry', 'Other', 'Paper Ephemera', 'Serving', 'Supplies', 'Toy', 'Trading Cards']


In [38]:
print(sorted(train[train['cat_1'] == 'Handmade']['cat_2'].unique()))

['Accessories', 'Art', 'Bags and Purses', 'Books and Zines', 'Candles', 'Ceramics and Pottery', 'Children', 'Clothing', 'Crochet', 'Dolls and Miniatures', 'Furniture', 'Geekery', 'Glass', 'Holidays', 'Housewares', 'Jewelry', 'Knitting', 'Music', 'Needlecraft', 'Other', 'Others', 'Paper Goods', 'Patterns', 'Pets', 'Quilts', 'Toys', 'Weddings', 'Woodworking']


In [39]:
print(sorted(train[train['cat_1'] == 'Sports & Outdoors']['cat_2'].unique()))

['Apparel', 'Exercise', 'Fan Shop', 'Footwear', 'Golf', 'Other', 'Outdoors', 'Team Sports']


In [40]:
# 대분류 카테고리 키워드 매핑
CATEGORY_KEYWORDS = {
    'Women': [
        'Athletic Apparel', 'Coats & Jackets', 'Dresses', 'Jeans', 'Jewelry',
        'Maternity', 'Other', 'Pants', 'Shoes', 'Skirts', 'Suits & Blazers',
        'Sweaters', 'Swimwear', 'Tops & Blouses', 'Underwear', "Women's Accessories", "Women's Handbags"
    ],
    'Beauty': [
        'Bath & Body', 'Fragrance', 'Hair Care', 'Makeup', 'Other', 'Skin Care', 'Tools & Accessories'
    ],
    'Kids': [
        'Bathing & Skin Care', 'Boys (4+)', 'Boys 0-24 Mos', 'Boys 2T-5T', 'Car Seats & Accessories',
        'Diapering', 'Feeding', 'Gear', 'Girls (4+)', 'Girls 0-24 Mos', 'Girls 2T-5T',
        'Health & Baby Care', 'Nursery', 'Other', 'Potty Training', 'Pregnancy & Maternity', 'Safety', 'Strollers', 'Toys'
    ],
    'Electronics': [
        'Cameras & Photography', 'Car Audio, Video & GPS', 'Cell Phones & Accessories', 'Computers & Tablets',
        'Media', 'Other', 'TV, Audio & Surveillance', 'Video Games & Consoles'
    ],
    'Men': [
        'Athletic Apparel', 'Blazers & Sport Coats', 'Coats & Jackets', 'Jeans', "Men's Accessories",
        'Other', 'Pants', 'Shoes', 'Shorts', 'Suits', 'Sweaters', 'Sweats & Hoodies', 'Swimwear', 'Tops'
    ],
    'Home': [
        'Artwork', 'Bath', 'Bedding', 'Cleaning Supplies', 'Furniture', 'Home Appliances', 'Home Décor',
        "Kids' Home Store", 'Kitchen & Dining', 'Other', 'Seasonal Décor', 'Storage & Organization'
    ],
    'Vintage & Collectibles': [
        'Accessories', 'Antique', 'Bags and Purses', 'Book', 'Clothing', 'Collectibles', 'Electronics',
        'Furniture', 'Home Decor', 'Housewares', 'Jewelry', 'Other', 'Paper Ephemera', 'Serving', 'Supplies', 'Toy', 'Trading Cards'
    ],
    'Handmade': [
        'Accessories', 'Art', 'Bags and Purses', 'Books and Zines', 'Candles', 'Ceramics and Pottery', 'Children', 'Clothing',
        'Crochet', 'Dolls and Miniatures', 'Furniture', 'Geekery', 'Glass', 'Holidays', 'Housewares',
        'Jewelry', 'Knitting', 'Music', 'Needlecraft', 'Other', 'Others', 'Paper Goods', 'Patterns', 'Pets',
        'Quilts', 'Toys', 'Weddings', 'Woodworking'
    ],
    'Sports & Outdoors': [
        'Apparel', 'Exercise', 'Fan Shop', 'Footwear', 'Golf', 'Other', 'Outdoors', 'Team Sports'
    ],
    'Other': []  # 위 어디에도 해당 안되면 Other
}

def infer_category(row):
    """name + item_description으로 카테고리 추론"""
    
    # category_name이 있으면 그대로 사용
    if pd.notnull(row['category_name']):
        return row['category_name']
    
    text = ' '.join([
        str(row['name']),
        str(row['item_description'])
    ]).lower()
    
    # 키워드 매칭으로 카테고리 추론
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return category  # 대분류만 반환 (세부분류 없음)
    
    return 'Other'

print("category_name 결측치 처리 중...")
train['category_name'] = train.apply(infer_category, axis=1)
test['category_name'] = test.apply(infer_category, axis=1)

# 결과 확인
print(f"처리 후 결측치 수: {train['category_name'].isna().sum()}")
print(f"\n추론된 카테고리 분포:")
print(train['category_name'].value_counts().head(15))

category_name 결측치 처리 중...
처리 후 결측치 수: 0

추론된 카테고리 분포:
category_name
Women/Athletic Apparel/Pants, Tights, Leggings                 60177
Women/Tops & Blouses/T-Shirts                                  46380
Beauty/Makeup/Face                                             34335
Beauty/Makeup/Lips                                             29910
Electronics/Video Games & Consoles/Games                       26557
Beauty/Makeup/Eyes                                             25215
Electronics/Cell Phones & Accessories/Cases, Covers & Skins    24676
Women/Underwear/Bras                                           21274
Women/Tops & Blouses/Tank, Cami                                20284
Women/Tops & Blouses/Blouse                                    20284
Women/Dresses/Above Knee, Mini                                 20082
Women/Jewelry/Necklaces                                        19758
Women/Athletic Apparel/Shorts                                  19528
Beauty/Makeup/Makeup Palettes      

In [42]:
# category_name이 채워졌으니 다시 분리
def split_category(category):
    if pd.isna(category):
        return pd.Series(['Other', 'Unknown', 'Unknown'])
    parts = category.split('/', 2)
    while len(parts) < 3:
        parts.append('Unknown')
    return pd.Series(parts[:3])

print("카테고리 재분리 중...")
train[['cat_1','cat_2','cat_3']] = train['category_name'].apply(split_category)
test[['cat_1','cat_2','cat_3']] = test['category_name'].apply(split_category)

# 최종 확인
print("\n=== 최종 cat_1 분포 ===")
print(train['cat_1'].value_counts().head(10))
print(f"\n결측치 완전 제거 확인:")
print(train[['category_name','cat_1','cat_2','cat_3']].isna().sum())


카테고리 재분리 중...

=== 최종 cat_1 분포 ===
cat_1
Women                     664385
Beauty                    207828
Kids                      171689
Electronics               122690
Men                        93680
Home                       67871
Other                      51678
Vintage & Collectibles     46530
Handmade                   30842
Sports & Outdoors          25342
Name: count, dtype: int64

결측치 완전 제거 확인:
category_name    0
cat_1            0
cat_2            0
cat_3            0
dtype: int64


#### 2-3. brand_name 결측치 확인 및 처리

In [57]:
# 전체 결측치 현황 한눈에 보기
print("=== 결측치 현황 ===")
print(train.isna().sum())
print(f"\nbrand_name 결측치 수: {train['brand_name'].isna().sum()}")
print(f"전체 대비 비율: {train['brand_name'].isna().mean()*100:.2f}%")

# 결측치 샘플 확인 - 어떤 상품들인지 보기
null_category = train[train['brand_name'].isna()][
    ['name', 'cat_1', 'cat_2', 'item_description', 'price']
].sample(20, random_state=42)

print("\n=== brand_name 결측 샘플 ===")
print(null_category.to_string())

=== 결측치 현황 ===
train_id                  0
name                      0
item_condition_id         0
category_name             0
brand_name           632682
price                     0
shipping                  0
item_description          6
cat_1                     0
cat_2                     0
cat_3                     0
dtype: int64

brand_name 결측치 수: 632682
전체 대비 비율: 42.68%

=== brand_name 결측 샘플 ===
                                             name        cat_1             cat_2                                                                                                                                                                                                                                                                                                                                                                                                                                                                    item_description  price
519130                  Perfume & neckl

##### 2-3-2. 카테고리별 서브데이터 분리

In [56]:
# 상위 카테고리 분포 확인 : 대분류('cat_1')의 경우, 총 10개의 카테고리와 결측치(6,327건)으로 구성
top10_categories = train['cat_1'].value_counts().head(10)
print(top10_categories)

cat_1
Women                     664385
Beauty                    207828
Kids                      171689
Electronics               122690
Men                        93680
Home                       67871
Other                      51678
Vintage & Collectibles     46530
Handmade                   30842
Sports & Outdoors          25342
Name: count, dtype: int64


In [48]:
# 상위 10개 카테고리 이름 추출
top10_cat_names = train['cat_1'].value_counts().head(10).index.tolist()
print("상위 10개 카테고리:", top10_cat_names)

# 딕셔너리로 서브데이터 생성
sub_dfs = {}

for cat in top10_cat_names:
    sub_dfs[cat] = train[train['cat_1'] == cat].copy()
    print(f"{cat}: {len(sub_dfs[cat])}행")

# 상위 10개에 포함되지 않는 나머지도 보존
sub_dfs['Others'] = train[~train['cat_1'].isin(top10_cat_names)].copy()
print(f"Others: {len(sub_dfs['Others'])}행")

상위 10개 카테고리: ['Women', 'Beauty', 'Kids', 'Electronics', 'Men', 'Home', 'Other', 'Vintage & Collectibles', 'Handmade', 'Sports & Outdoors']
Women: 664385행
Beauty: 207828행
Kids: 171689행
Electronics: 122690행
Men: 93680행
Home: 67871행
Other: 51678행
Vintage & Collectibles: 46530행
Handmade: 30842행
Sports & Outdoors: 25342행
Others: 0행


In [58]:
print(train['category_name'].value_counts().head(25))

category_name
Women/Athletic Apparel/Pants, Tights, Leggings                 60177
Women/Tops & Blouses/T-Shirts                                  46380
Beauty/Makeup/Face                                             34335
Beauty/Makeup/Lips                                             29910
Electronics/Video Games & Consoles/Games                       26557
Beauty/Makeup/Eyes                                             25215
Electronics/Cell Phones & Accessories/Cases, Covers & Skins    24676
Women/Underwear/Bras                                           21274
Women/Tops & Blouses/Tank, Cami                                20284
Women/Tops & Blouses/Blouse                                    20284
Women/Dresses/Above Knee, Mini                                 20082
Women/Jewelry/Necklaces                                        19758
Women/Athletic Apparel/Shorts                                  19528
Beauty/Makeup/Makeup Palettes                                  19103
Women/Shoes/Boots   

##### 2-3-3. 대분류(cat_1) 각각 속한 브랜드 출력

In [99]:
print(train['brand_name'].value_counts().head(27))

brand_name
Unknown              119156
M                     91460
PINK                  64480
Nike                  55786
LuLaRoe               50781
Victoria's Secret     48637
All                   37460
DL                    26125
Apple                 24023
GH                    16670
Op                    15956
FOREVER 21            15555
GE                    15463
Nintendo              15132
Michael Kors          14867
Lululemon             14613
American Eagle        14059
Disney                13714
Adidas                13073
Rae Dunn              12991
Ash                   12951
Coach                 12425
Sephora               12216
Com                   11660
SO                    10998
Ice                   10870
Funko                 10821
Name: count, dtype: int64


In [67]:
print(sorted(sub_dfs['Women']['brand_name'].unique()))

['!iT Jeans', '191 Unlimited', '21men', '24/7 Comfort Apparel', '2XU', '3.1 Phillip Lim', '47 Brand', '5.11 Tactical', '525 America', '5th & Ocean', '7 For All Mankind®', '90 Degree By Reflex', 'A Bathing Ape', 'A Pea In The Pod', 'A Plus Child Supply', 'A Wish Come True', 'A&A Optical', 'A&E', 'A&R Sports', 'A+D', 'A. Byer', 'A.B.S. by Allen Schwartz', 'A.K.A', 'A.L.C.', 'A.P.C.', 'A/X Armani Exchange', 'AB Studio', 'ABS by Allen Schwartz', 'ADAM', 'AG Adriano Goldschmied', 'AGB', 'AGUADECOCO', 'AKIRA', 'ALDO', 'ALEX AND ANI', 'ALLOY', 'ALO Yoga', 'ANAMA', 'ANDREA FENZI', 'ANGL', 'ARCONA', 'ASICS', 'ASOS', 'ASSETS by Sara Blakely', 'ASTR', 'AX Paris', 'Abercrombie & Fitch', 'Acacia Swimwear', 'Acne Jeans', 'Acorn', 'Acrobat', 'Active', 'Add Down', 'Adee Kaye', 'Adidas', 'Adonna', 'Adrianna Papell', 'Adrienne Landau', 'Adrienne Vittadini', 'Aerie', 'Aero', 'Aeropostale', 'Aerosoles', 'Affliction', 'Agent Provocateur', 'Agnes & Dora', 'Agua Bendita', 'Aidan Mattox', 'Air Force', 'Air Jo

In [68]:
print(sorted(sub_dfs['Beauty']['brand_name'].unique()))

['% Pure', 'A Wish Come True', 'A. Byer', 'AERIN', 'Abercrombie & Fitch', 'Acure', 'Adidas', 'Air Wick', 'Alexandra de Markoff', 'Algenist', 'Almay', 'Amazon', 'Ambi', 'American Eagle', 'Anastasia Beverly Hills', 'Andis', 'Anthropologie', 'Apple', 'Arbonne', 'Ardell', 'Ariana Grande', 'Artist Couture', 'As Seen on TV', 'Aussie', 'Australian Gold', 'Aveda', 'Aveeno', 'Avon', 'Axe', 'Azzaro', 'BH Cosmetics', 'Baby Phat', 'Balmain', 'Banana Boat', 'Bare Escentuals', 'Bare Minerals', 'Bareminerals', 'Bass', 'Bath & Body Works', 'BeautiControl', 'Beautycounter', 'Bebe', 'Becca', 'Becca Cosmetics', 'BedHead', 'Befine', 'Belif', 'Ben Nye', 'Benefit', 'Benefit Cosmetics', 'Better Shea Butter', 'Bh', 'Bio Oil', 'Biore', 'Bite Beauty', 'Blinc', 'Blink', 'Bliss', 'Bobbi Brown', 'Bonne Bell', 'Borghese', 'Boscia', 'Bourjois', 'Braun', 'Briogeo', 'Brut', 'Bumble and bumble', 'Burberry', 'Burberry London', "Burt's Bees", 'Buxom', 'Bvlgari', 'By Terry', 'Bésame Cosmetics', 'C. O. Bigelow', 'CHI', 'CI

In [69]:
print(sorted(sub_dfs['Kids']['brand_name'].unique()))

['A Bathing Ape', 'A Plus Child Supply', 'A Wish Come True', 'A+D', 'A-Shirt', 'A.D. Sutton & Sons', 'ABC Studios', 'AKA New York', 'ART', 'ASICS', 'ASUS', 'Abercrombie & Fitch', 'Accessory Innovations', 'Accessory Workshop', 'Activision', 'Adidas', 'Adora', 'Aero', 'Aeropostale', 'Affliction', 'Agetec', 'Air Hogs', 'Air Jordan', 'Air Zone', 'Akai', 'Alex Toys', 'All', 'Amazon', 'Ameda', 'American Boy & Girl', 'American Girl ®', 'American apparel', 'Angelcare', 'Angry Birds', 'Animal Adventure', 'Animal Alley', 'Animal Planet', 'Anita', 'Antique Rivet', 'Apple', 'Aquaphor', 'Aquarius', 'Aquatopia', 'Ariat', 'Arizona', 'Arizona Jean Company', 'Arm & Hammer', "Arm's Reach Concepts", 'Armani', 'As Seen on TV', 'Aspeed', 'Atari', 'Athletech', 'Athletic Works', 'Atlantis Toy and Hobby', 'Aurora World', 'Aveeno', 'Avent', 'Avon', 'B is for Bear', 'BOB Strollers', 'BOGS', 'BOOMco.', 'Babies R Us', 'Babies R Us Infant', 'Babies R Us Plush', 'Baby Aspen', 'Baby Boom', 'Baby Brezza', 'Baby Bulle

In [70]:
print(sorted(sub_dfs['Electronics']['brand_name'].unique()))

['1byone', '2K Games', '3M®', 'A Bathing Ape', 'A&A Optical', 'ACER', 'AMD', 'ASUS', 'AT&T', 'ATI Technologies', 'Accessory Collective', 'Acer', 'Activision', 'Adidas', 'Advent', 'Agfa', 'Aiptek', 'Air Hogs', 'Alienware', 'Alpine', 'Altec Lansing', 'Amazon', 'AmazonBasics', 'American Girl ®', 'American apparel', 'Anker', 'Apple', 'Astro', 'Atari', 'Audio Technica', 'Audio-Technica', 'AudioSource', 'Audiovox', 'Aurum', 'Avery', 'Avon', 'BLUE', 'Bandai', 'Bang & Olufsen', 'Barbie', 'Bass', 'Beats', 'Beats by Dr. Dre', 'Belkin', 'Black & Decker', 'Blizzard', 'Body Glove', 'Bose', 'Brother', 'Bushnell', 'Cannon', 'Canon', 'Capcom', 'Carhartt', 'Case Logic', 'Chloe', 'Chromo Inc', 'Coach', 'Coby Electronics', 'Coleman', 'Com', 'Compaq', 'Conair', 'Converse', 'Cooler Master', 'Corsair', 'Crayola', 'Creative Labs', 'Crosley Radio', 'Crucial', 'Curtis Computer Products', 'Custom Accessories', 'D-Link', 'DC Comics', 'DIRECTV', 'DJI', 'DVS', 'DYMO', 'Dazzle Multimedia', 'Dell', 'Denon', 'Dior', 

In [71]:
print(sorted(sub_dfs['Men']['brand_name'].unique()))

['10.Deep', '21men', '24/7 Comfort Apparel', '47', '47 Brand', '5.11 Tactical', '7 Diamonds', '7 For All Mankind®', '8732 Apparel', 'A Bathing Ape', 'A&A Optical', 'A&R Sports', 'A+D', 'A-Shirt', 'A.K.A', 'A.P.C.', 'A/X Armani Exchange', 'AC/DC', 'ADAM', 'AG Adriano Goldschmied', 'AKOO', 'ALDO', 'AND', 'AND1', 'ASICS', 'ASOLO', 'ASOS', 'Abercrombie & Fitch', 'Academy', 'Accutron', 'Acne Jeans', 'Acne Studios', 'Active', 'Activision', 'Addison', 'Adidas', 'Adolfo', 'Aeropostale', 'Affliction', 'Air Force', 'Air Jordan', 'Airwalk', 'Akademiks', 'Alan Flusser', 'Alexander Julian', 'Alexander McQueen', 'Alfani', 'Alien Workshop', 'Alife', 'All Saints', 'All Sport', 'AllSaints', 'Allen Edmonds', 'Allen Solly', 'Alpha Industries', 'Alpine Design', 'Alpinestars', 'Alstyle Apparel', 'Altamont', 'Alternative', 'American Apparel', 'American Classics', 'American Eagle', 'American Fighter', 'American Needle', 'American Rag', 'American Tourister', 'American Vintage', 'American apparel', 'Amphibious

In [72]:
print(sorted(sub_dfs['Home']['brand_name'].unique()))

['A Bathing Ape', 'A Pea In The Pod', 'A&A Optical', 'A. Byer', 'AMIA', 'Abbott', 'Adagio', 'Adidas', 'Advanced Graphics', 'AeroLatte', 'AeroPress', 'Agraria', 'Air Wick', 'Alessi', 'Alexander Kalifano', 'All', 'All American', 'All-Clad', 'Alpine', 'Amazon', 'American Girl ®', 'American Vintage', 'American Weigh', 'American apparel', 'Anchor Hocking', 'Anne Geddes', 'Anolon', 'Anthropologie', 'Apple', 'Aria', 'Arm & Hammer', 'Aroma', 'As Seen on TV', 'Aura Cacia', 'Avon', 'Avon Cosmetics, Inc', 'BUNN', 'Back to Basics', 'Ball', 'Baratza', 'Barbie', 'Bass', 'Bath & Body Works', 'Bella Cucina', 'Betsey Johnson', 'Betty Boop', 'Betty Crocker', 'Bialetti', 'Bissell', 'Black & Decker', 'Blendtec', 'Blizzard', 'Bloomfield', 'Blossom Bucket', 'Blunt Power', 'Bodum', 'Bosch', 'Boy + Girl', 'Braun', 'Brentwood', 'Breville', 'Breyer', 'Brighton', 'Brita', 'Brookstone', 'Bulova', 'CBK', 'CORKCICLE', 'Calphalon', 'Calvin Klein', 'Candle-Lite', 'Cannon', 'Capresso', 'Carhartt', 'Carson Home Accents

In [73]:
print(sorted(sub_dfs['Vintage & Collectibles']['brand_name'].unique()))

['7 For All Mankind®', 'A Wish Come True', 'A. Byer', 'AC/DC', 'ADAM', 'ALDO', 'ALEX AND ANI', 'ASICS', 'ASOS', 'Abercrombie & Fitch', 'Acacia Swimwear', 'Accessory Power', 'Activision', 'Adidas', 'Adolfo', 'Adrianna Papell', 'Adrienne Vittadini', 'Aeropostale', 'Agent Provocateur', 'Air Force', 'Air Hogs', 'Air Jordan', 'Alexander McQueen', 'Alfred Angelo', 'Alfred Dunner', 'Alice + Olivia', 'AllSaints', 'Almost Famous', 'Alternative', 'Amazon', 'American Apparel', 'American Boy & Girl', 'American Eagle', 'American Girl ®', 'American Rag', 'American apparel', 'Anchor Hocking', 'Andrea by Sadek', 'Angels', 'Angry Birds', 'Ann Taylor LOFT', 'Anne Klein', 'Anthropologie', 'Antique Rivet', 'Antonio Melani', 'Anvil', 'Apple', 'Apple Bottoms', 'Apt.', 'Aquarius', 'Arizona', 'Army', 'Athleta', 'Atmosphere', 'Avanti', 'Avenue', 'Avery', 'Avon', 'A|X Armani Exchange', 'B. Darlin', 'BCBGMAXAZRIA', 'BCBGeneration', 'BDG', 'BOSS HUGO BOSS', 'Baby Phat', 'Bachrach', 'Back to Basics', 'Badgley Misc

In [74]:
print(sorted(sub_dfs['Other']['brand_name'].unique()))

['3M®', '7 For All Mankind®', 'A Bathing Ape', 'A Pea In The Pod', 'A+D', 'A. Byer', 'AA Aquarium', 'ACCEL', 'ACDelco', 'AG Adriano Goldschmied', 'API', 'ASICS', 'ASUS', 'AT-A-GLANCE', 'Abbott', 'Abercrombie & Fitch', 'Able provider', 'Act', 'Actron®', 'Adams', 'Adidas', 'Advanced Healthcare Distributors', 'Advantage', 'Advil', 'Aeropostale', 'Affliction', 'Air Jordan', 'Akai', 'Alcon', 'Alfred Dunner', 'All', 'All Four Paws', 'All Living Things', 'Almay', 'Almost Famous', "Altar'd State", 'Always', 'Amazon', 'Amazon Essentials', 'Ambiance Apparel', 'American Apparel', 'American Eagle', 'American Fighter', 'American Girl ®', 'American Rag', 'American apparel', 'Amsoil', 'Anastasia Beverly Hills', 'Andis', 'Andrew Christian', 'Angels', 'Ann Taylor', 'Ann Taylor LOFT', 'Anne Michelle', 'Anthropologie', 'Apple', 'Aqua Clear', 'Aquafresh', 'Aquaphor', 'Aqueon', 'Arbonne', 'Archie Comics', 'Ardell', 'Arden B', 'Arizona Jean Company', 'Arm & Hammer', 'Armani Jeans', 'Armor All®', 'Army', 'Ar

In [75]:
print(sorted(sub_dfs['Handmade']['brand_name'].unique()))

['A Bathing Ape', 'Adidas', 'American Girl ®', 'American apparel', 'Apple', 'Audi', 'Avon', 'Bass', 'Betsey Johnson', 'Birkenstock', 'Blue Buffalo', 'Brandy Melville', 'Calvin Klein', 'Chloe', 'Coach', 'Colorbok', 'Converse', 'Custom Accessories', 'Customized & Personalized', 'DC Comics', "David's Bridal", 'Dior', 'Disney', 'Dooney & Bourke', 'EK Success', 'Elmers', 'Faber-Castell', 'Fila', 'Funimation Production', 'Gucci', 'Guy Harvey', 'H&M', 'Hallmark', 'Hello Kitty', 'Hollister', 'Hot Topic', 'Independent', 'JanSport', 'Jordan', 'Kate Spade', 'Kylie Cosmetics', 'LuLaRoe', 'MARC JACOBS', 'MLB', 'Martha Stewart', 'Medela', 'Michael Kors', 'Mighty Fine', 'Minnie Mouse', 'Nabisco Variety', 'Nike', 'North Face', 'PINK', 'Philadelphia Eagles', 'Pokemon', 'Rae Dunn', 'Ralph Lauren', 'Rowan', 'Samsung', 'Serenity', 'Sesame Street', 'Spiderman', 'Spin Master', 'Supreme', 'Swarovski', 'TOMS', 'Target', 'Tommy Hilfiger', 'Under Armour', 'Unknown', 'Vans', 'Vantage', 'Victoria secret', "Victor

In [76]:
print(sorted(sub_dfs['Sports & Outdoors']['brand_name'].unique()))

['5.11 Tactical', 'A Bathing Ape', 'AND1', 'ASICS', 'Abu Garcia', 'Adams Golf', 'Adidas', 'Airwalk', 'Alstyle Apparel', 'American apparel', 'Apple', 'Aqua Lung Sport', 'Arena', 'As Seen on TV', 'Asics', 'Avia', 'BLACKHAWK!', 'BLOCH', 'BOGS', 'Babolat', 'Barnett Crossbows', 'Barska', 'Bass', 'Bauer', 'Bear Archery', 'Bebe', 'Berkley', 'Betsey Johnson', 'Blitz®', 'Bowflex', 'Bridgestone Golf', 'Brine', 'Brooks', 'Browning', 'Brute', 'Burton', 'Bushnell', 'CCM', 'CORKCICLE', 'Callaway', 'CamelBak', 'Capezio', 'Carhartt', 'Cascade', 'Century', 'Champion', 'Champro', 'Cleveland', 'Cleveland Golf', 'Cliff Keen', 'Coach', 'Coast', 'Cobra', 'Coleman', 'Columbia', 'Comfort Zone', 'Contigo', 'Converse Shoes', 'Crocs', 'Crosman', 'Cutters', 'DBX', 'DC Shoes', 'DC Sports', 'Danskin Now', 'DeMarini', 'Diadora', 'Dior', 'Disney', 'Dragon', 'Dunlop', 'Easton', 'Eddie Bauer', 'Ektelon', 'Everlast', 'EvoShield', 'Faded Glory', 'Field & Stream', 'Fila', 'Fitbit', 'Fitness Gear', 'Flow Society', 'Focus',

##### 2-3-4. 'brand_name' 결측치 처리 : 대분류(cat_1) 각각 속한 브랜드 자동출력

In [77]:
# ==============================================
# Step 1: 카테고리별 실제 브랜드 리스트 자동 추출
# ==============================================

# 현재 존재하는 모든 cat_1 카테고리 확인
all_categories = train['cat_1'].unique()
print("전체 카테고리 목록:")
print(sorted(all_categories))

# 카테고리별 브랜드 딕셔너리 자동 생성
AUTO_CATEGORY_BRANDS = {}

for cat in all_categories:
    # Unknown 제외하고 실제 브랜드명만 추출
    brands = sorted(
        sub_dfs[cat]['brand_name'].dropna().unique().tolist()
        if cat in sub_dfs
        else train[train['cat_1'] == cat]['brand_name'].dropna().unique().tolist()
    )
    # Unknown 제거
    brands = [b for b in brands if b not in ['Unknown', 'unknown', '', 'nan']]
    AUTO_CATEGORY_BRANDS[cat] = brands
    print(f"{cat}: {len(brands)}개 브랜드 추출")

print("\n추출 완료!")

전체 카테고리 목록:
['Beauty', 'Electronics', 'Handmade', 'Home', 'Kids', 'Men', 'Other', 'Sports & Outdoors', 'Vintage & Collectibles', 'Women']
Men: 1220개 브랜드 추출
Electronics: 343개 브랜드 추출
Women: 2541개 브랜드 추출
Home: 427개 브랜드 추출
Sports & Outdoors: 292개 브랜드 추출
Vintage & Collectibles: 694개 브랜드 추출
Beauty: 516개 브랜드 추출
Other: 1002개 브랜드 추출
Kids: 932개 브랜드 추출
Handmade: 79개 브랜드 추출

추출 완료!


In [100]:
# ==============================================
# Step 2: 카테고리별 브랜드 확인 (검증용)
# ==============================================

# 특정 카테고리 브랜드 출력해서 확인
for cat in all_categories:
    print(f"\n=== {cat} 브랜드 샘플 (상위 25개) ===")
    print(AUTO_CATEGORY_BRANDS[cat][:25])


=== Men 브랜드 샘플 (상위 25개) ===
['10.Deep', '21men', '24/7 Comfort Apparel', '47', '47 Brand', '5.11 Tactical', '7 Diamonds', '7 For All Mankind®', '8732 Apparel', 'A Bathing Ape', 'A&A Optical', 'A&R Sports', 'A+D', 'A-Shirt', 'A.K.A', 'A.P.C.', 'A/X Armani Exchange', 'AC/DC', 'ADAM', 'AG Adriano Goldschmied', 'AKOO', 'ALDO', 'AND', 'AND1', 'ASICS']

=== Electronics 브랜드 샘플 (상위 25개) ===
['1byone', '2K Games', '3M®', 'A Bathing Ape', 'A&A Optical', 'ACER', 'AMD', 'ASUS', 'AT&T', 'ATI Technologies', 'Accessory Collective', 'Acer', 'Activision', 'Adidas', 'Advent', 'Agfa', 'Aiptek', 'Air Hogs', 'Alienware', 'Alpine', 'Altec Lansing', 'Amazon', 'AmazonBasics', 'American Girl ®', 'American apparel']

=== Women 브랜드 샘플 (상위 25개) ===
['!iT Jeans', '191 Unlimited', '21men', '24/7 Comfort Apparel', '2XU', '3.1 Phillip Lim', '47 Brand', '5.11 Tactical', '525 America', '5th & Ocean', '7 For All Mankind®', '90 Degree By Reflex', 'A Bathing Ape', 'A Pea In The Pod', 'A Plus Child Supply', 'A Wish Come T

In [101]:
# 모든 카테고리에 공통으로 적용할 브랜드
COMMON_BRANDS = [
    '3M®','7 For All Mankind®', "A.P.C.", 'ALDO','ASICS','A&A Optical','Amazon','AmazonBasics',
    '21men','A+D', 'A&E', 'A Bathing Ape', 'A Pea In The Pod','A. Byer','Abercrombie & Fitch',
    'PINK', 'Nike', 'Fila', 'Under Armour','Adidas', 'Apple', 'Samsung', 'MLB', 'Avon', 'Air Jordan',
    'Target', 'Converse', 'Disney','North Face', 'UNIQLO', 'Zara', 'Gucci', 'A Wish Come True',
    'Vans','Supreme', 'TOMS', 'Ralph Lauren','jansport','Chloe','Michael Kors','H&M','Carhartt',
    'CORKCICLE', 'Yeti', 'Tommy Hilfiger', 'Victoria secret','Dior',"LuLaRoe",'Birkenstock','Joyfolie',
    'Jordan','Dolce & Gabana','Guy Harvey', 'Sons of Anarchy','Hollister','Pokemon','Philadelphia Eagles',
    'Bass','doTERRA','American apparel','MARC JACOBS','Kylie Cosmetics'
    ]

In [102]:
# ==============================================
# Step 3: 브랜드 매칭 함수 정의
# ==============================================

def extract_brand_auto(row, brand_list, common_brands=COMMON_BRANDS):
    """
    데이터에서 추출한 실제 브랜드 리스트로 결측치 채우기
    우선순위: 기존 브랜드 유지 > 카테고리 브랜드 매칭 > 공통 브랜드 매칭 > Unknown
    """
    # 이미 브랜드가 있으면 유지
    if row['brand_name'] not in ['Unknown', 'unknown', '', 'nan'] and pd.notnull(row['brand_name']):
        return row['brand_name']
    
    # 텍스트 준비 (name + item_description)
    text = ' '.join([
        str(row['name']),
        str(row['item_description'])
    ]).lower()
    
    # 카테고리 특화 브랜드 먼저 탐색
    for brand in brand_list:
        if str(brand).lower() in text:
            return brand
    
    # 공통 브랜드 탐색
    for brand in common_brands:
        if str(brand).lower() in text:
            return brand
    
    return 'Unknown'

In [103]:
# ==============================================
# Step 4: 카테고리별 브랜드 결측치 채우기 적용
# ==============================================

print("브랜드 결측치 자동 채우기 시작...\n")

# train 처리
for cat in all_categories:
    brand_list = AUTO_CATEGORY_BRANDS.get(cat, [])
    mask = train['cat_1'] == cat
    
    train.loc[mask, 'brand_name'] = (
        train[mask].apply(
            lambda row: extract_brand_auto(row, brand_list), axis=1
        )
    )
    
    # 진행상황 출력
    filled_rate = (train.loc[mask, 'brand_name'] != 'Unknown').mean() * 100
    print(f"  {cat:30s}: {filled_rate:.1f}% 브랜드 확인")

# test 처리 (train에서 추출한 브랜드 리스트 동일 적용)
print("\ntest 데이터 처리 중...")
for cat in all_categories:
    brand_list = AUTO_CATEGORY_BRANDS.get(cat, [])
    mask = test['cat_1'] == cat
    
    if mask.sum() == 0:  # 해당 카테고리가 test에 없으면 스킵
        continue
        
    test.loc[mask, 'brand_name'] = (
        test[mask].apply(
            lambda row: extract_brand_auto(row, brand_list), axis=1
        )
    )

브랜드 결측치 자동 채우기 시작...

  Men                           : 95.2% 브랜드 확인
  Electronics                   : 94.0% 브랜드 확인
  Women                         : 96.2% 브랜드 확인
  Home                          : 92.5% 브랜드 확인
  Sports & Outdoors             : 76.4% 브랜드 확인
  Vintage & Collectibles        : 94.7% 브랜드 확인
  Beauty                        : 96.3% 브랜드 확인
  Other                         : 71.7% 브랜드 확인
  Kids                          : 85.8% 브랜드 확인
  Handmade                      : 28.8% 브랜드 확인

test 데이터 처리 중...


In [104]:
# ==============================================
# Step 5: 최종 결과 확인
# ==============================================

train_final_rate = (train['brand_name'] != 'Unknown').mean() * 100
test_final_rate  = (test['brand_name'] != 'Unknown').mean() * 100

print(f"\n{'='*40}")
print(f"train 최종 브랜드 확인율: {train_final_rate:.1f}%")
print(f"test  최종 브랜드 확인율: {test_final_rate:.1f}%")
print(f"{'='*40}")

# 카테고리별 최종 확인율 정리
print("\n=== 카테고리별 최종 브랜드 확인율 ===")
for cat in all_categories:
    mask = train['cat_1'] == cat
    rate = (train.loc[mask, 'brand_name'] != 'Unknown').mean() * 100
    bar  = '█' * int(rate // 5)  # 간단한 바 차트
    print(f"  {cat:30s}: {rate:5.1f}% {bar}")



train 최종 브랜드 확인율: 92.0%
test  최종 브랜드 확인율: 92.0%

=== 카테고리별 최종 브랜드 확인율 ===
  Men                           :  95.2% ███████████████████
  Electronics                   :  94.0% ██████████████████
  Women                         :  96.2% ███████████████████
  Home                          :  92.5% ██████████████████
  Sports & Outdoors             :  76.4% ███████████████
  Vintage & Collectibles        :  94.7% ██████████████████
  Beauty                        :  96.3% ███████████████████
  Other                         :  71.7% ██████████████
  Kids                          :  85.8% █████████████████
  Handmade                      :  28.8% █████


In [105]:
# Unknown 샘플 확인 - 왜 브랜드를 못 찾았는지 살펴보기
unknown_samples = train[train['brand_name'] == 'Unknown'][
    ['name', 'item_description', 'cat_1']
].sample(25, random_state=42)

print(unknown_samples.to_string())

                                    name                                                                                                                                          item_description        cat_1
359525                            Blouse                                                                                                                                                Super cute        Women
600354                             1 box                                                                                                                                      35$ shipping 600 box        Other
228112            Boys Sweatpants Bundle                                                                                                                                                       GUC         Kids
479825           [rm] Best Buy Gift Card                                                                         This is a brand new, never been used [rm] Best Buy gift

In [107]:
# 팬덤/캐릭터 키워드 피처 추가
FANDOM_KEYWORDS = [
    'BTS', 'Kpop', 'K-pop', 'Anime', 'Disney', 'Marvel', 'DC',
    'Harry Potter', 'Star Wars', 'Pokemon', 'Panic At The Disco',
    'Taylor Swift', 'Ariana Grande', 'Kendall', 'Kylie', 'The Vamps', 'Ren and Stimpy',
    'hello kitty','Mickey Mouse', 'The Beatles'
]

def has_fandom(text):
    text = str(text)
    return int(any(kw.lower() in text.lower() for kw in FANDOM_KEYWORDS))

train['is_fandom'] = (
    train['name'] + ' ' + train['item_description']
).apply(has_fandom)

print(f"팬덤 상품 수: {train['is_fandom'].sum()}")

팬덤 상품 수: 50268


In [108]:
train['brand_name'].nunique()

4883

In [109]:
print(train['brand_name'].value_counts().head(25))  # 전체 상위 브랜드 확인

brand_name
Unknown              119076
M                     91460
PINK                  64480
Nike                  55786
LuLaRoe               50781
Victoria's Secret     48637
All                   37460
DL                    26125
Apple                 24023
GH                    16670
Op                    15956
FOREVER 21            15555
GE                    15463
Nintendo              15132
Michael Kors          14867
Lululemon             14613
American Eagle        14059
Disney                13714
Adidas                13073
Rae Dunn              12991
Ash                   12951
Coach                 12425
Sephora               12216
Com                   11660
SO                    10998
Name: count, dtype: int64


'brand_name' = 'M' 확인 필요

In [110]:
train[train['brand_name']=='M']

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,cat_1,cat_2,cat_3,is_fandom
9,9,Porcelain clown doll checker pants VTG,3,Vintage & Collectibles/Collectibles/Doll,M,8.0,0,I realized his pants are on backwards after th...,Vintage & Collectibles,Collectibles,Doll,0
42,42,lots of Korean Nature Republic face mask,1,Beauty/Skin Care/Face,M,14.0,1,"totally 36 masks, will be expired on Feb..",Beauty,Skin Care,Face,0
43,43,Apricot beige stick foundation!!,1,Beauty/Makeup/Face,M,12.0,1,Great quality!!! Fast free shipping!! You can ...,Beauty,Makeup,Face,0
44,44,Glass Christmas Bowl✨,2,Vintage & Collectibles/Collectibles/Glass,M,12.0,1,Brand new! Never used smoking bowl. Just bough...,Vintage & Collectibles,Collectibles,Glass,0
49,49,Younique 3d fiber lash mascara,1,Beauty/Makeup/Eyes,M,9.0,1,Younique 3d fiber lash mascara will quickly be...,Beauty,Makeup,Eyes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1482407,1482407,Slim extreme fat burner serum,1,Beauty/Skin Care/Body,M,16.0,0,Firming Fighting cellulite Sliming Body contou...,Beauty,Skin Care,Body,0
1482466,1482466,For V,1,Home/Artwork/Paintings,M,31.0,0,"Painting With signature added 16x20 Canvas, ha...",Home,Artwork,Paintings,0
1482498,1482498,Cupquake beauty rush mist ON HOLD,3,Beauty/Fragrance/Women,M,27.0,0,No description yet,Beauty,Fragrance,Women,0
1482501,1482501,Jeffrey Star Velour Liquid Lipstick,2,Beauty/Makeup/Lips,M,38.0,0,Hey so I ordered these online and finally got ...,Beauty,Makeup,Lips,0


### 3. 텍스트 핵심 키워드 추출 (자연어 정리) : TF-IDF 방식 사용

In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# 불용어 설정 (영어 기본 + 커스텀 추가)
stop_words = set(stopwords.words('english'))
custom_stops = {'item', 'please', 'great', 'good', 'nice', 'make', 'used', 'new'}
stop_words.update(custom_stops)

def clean_text(text):
    """텍스트 정제: 소문자화, 특수문자 제거, 불용어 제거"""
    if pd.isnull(text):
        return 'unknown'
    
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # 특수문자 제거
    
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    
    return ' '.join(tokens)

# 텍스트 정제 적용
train['clean_desc'] = train['item_description'].apply(clean_text)
train['clean_name'] = train['name'].apply(clean_text)
test['clean_desc']  = test['item_description'].apply(clean_text)
test['clean_name']  = test['name'].apply(clean_text)

# name + description 합치기
train['text_combined'] = train['clean_name'] + ' ' + train['clean_desc']
test['text_combined']  = test['clean_name']  + ' ' + test['clean_desc']

# TF-IDF 벡터화 (상위 50,000 단어만 사용)
tfidf_desc = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))

X_train_text = tfidf_desc.fit_transform(train['text_combined'])
X_test_text  = tfidf_desc.transform(test['text_combined'])

print(f"TF-IDF 행렬 크기: {X_train_text.shape}")

TF-IDF 행렬 크기: (1482535, 50000)


### 4. 제품 특징 feature 생성 : 색깔, 소재

In [41]:
# 색상 키워드
COLORS = ['red','blue','green','black','white','pink','purple',
          'yellow','orange','grey','gray','brown','gold','silver']

# 소재 키워드
MATERIALS = ['cotton','leather','silk','wool','polyester','nylon',
             'denim','linen','velvet','suede','rubber','metal']

def extract_features(text):
    """텍스트에서 색상, 소재 추출"""
    text = str(text).lower()
    colors    = [c for c in COLORS    if c in text]
    materials = [m for m in MATERIALS if m in text]
    return (
        colors[0]    if colors    else 'unknown',
        materials[0] if materials else 'unknown'
    )

train[['color', 'material']] = train['text_combined'].apply(
    lambda x: pd.Series(extract_features(x))
)
test[['color', 'material']] = test['text_combined'].apply(
    lambda x: pd.Series(extract_features(x))
)

# 묶음/개수 추출 (예: "set of 3", "bundle", "lot of")
def has_bundle(text):
    text = str(text).lower()
    return int(any(w in text for w in ['bundle', 'set of', 'lot of', 'pack of', 'set']))

train['is_bundle'] = train['text_combined'].apply(has_bundle)
test['is_bundle']  = test['text_combined'].apply(has_bundle)

### 5-6. 모델 학습 및 가격 예측

In [42]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

# 레이블 인코딩 (카테고리형 변수)
cat_cols = ['cat_1', 'cat_2', 'cat_3', 'brand_name',
            'color', 'material']

le = LabelEncoder()
for col in cat_cols:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col + '_enc'] = le.transform(train[col].astype(str))
    test[col  + '_enc'] = le.transform(test[col].astype(str))

# 수치형 피처 선택
num_features = ['item_condition_id', 'shipping', 'is_bundle',
                'cat_1_enc', 'cat_2_enc', 'cat_3_enc',
                'brand_name_enc', 'color_enc', 'material_enc']

X_train_num = csr_matrix(train[num_features].values)
X_test_num  = csr_matrix(test[num_features].values)

# TF-IDF + 수치형 합치기
X_train_all = hstack([X_train_text, X_train_num])
X_test_all  = hstack([X_test_text,  X_test_num])

# 타겟: 가격 로그 변환 (왜도 줄이기)
y_train = np.log1p(train['price'])

# 학습/검증 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_all, y_train, test_size=0.1, random_state=42
)

# LightGBM 모델 학습
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 127,
    'n_estimators': 1000,
    'min_child_samples': 20,
    'random_state': 42
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# 예측
pred_log = model.predict(X_test_all)
pred_price = np.expm1(pred_log)  # 로그 역변환
pred_price = np.clip(pred_price, 0, None)  # 음수 방지

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
submission['price'] = pred_price
submission.to_csv('my_submission.csv', index=False)
print("제출 파일 생성 완료!")
print(submission.head())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 105.385548 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1123105
[LightGBM] [Info] Number of data points in the train set: 1334281, number of used features: 50006
[LightGBM] [Info] Start training from score 2.978544
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.507799
[200]	valid_0's rmse: 0.486044
[300]	valid_0's rmse: 0.476043
[400]	valid_0's rmse: 0.469779
[500]	valid_0's rmse: 0.465474
[600]	valid_0's rmse: 0.462057
[700]	valid_0's rmse: 0.459193
[800]	valid_0's rmse: 0.457182
[900]	valid_0's rmse: 0.455465
[1000]	valid_0's rmse: 0.453972
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 0.453972


c:\Users\berga\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


제출 파일 생성 완료!
   test_id      price
0        0   7.548861
1        1  11.036666
2        2  33.649194
3        3  13.766893
4        4   6.926652
